**MODELING**

In [31]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns
import ast

In [32]:
LABEL_NAMES = {
    0: "amusement",
    1: "excitement",
    2: "joy",
    3: "love",
    4: "desire",
    5: "optimism",
    6: "caring",
    7: "pride",
    8: "admiration",
    9: "gratitude",
    10: "relief",
    11: "approval",
    12: "realization",
    13: "surprise",
    14: "curiosity",
    15: "confusion",
    16: "fear",
    17: "nervousness",
    18: "remorse",
    19: "embarrassment",
    20: "disappointment",
    21: "sadness",
    22: "grief",
    23: "disgust",
    24: "anger",
    25: "annoyance",
    26: "disapproval",
    27: "neutral"
}

In [33]:
from pathlib import Path
DATA_DIR = Path('../data/raw')

train_df = pd.read_csv(DATA_DIR / "train.csv", encoding='utf-8', encoding_errors='replace')
val_df = pd.read_csv(DATA_DIR / "val.csv", encoding='utf-8', encoding_errors='replace')
test_df = pd.read_csv(DATA_DIR / "test.csv", encoding='utf-8', encoding_errors='replace')

train_df["labels"] = train_df["labels"].apply(ast.literal_eval)
val_df["labels"] = val_df["labels"].apply(ast.literal_eval)
test_df["labels"] = test_df["labels"].apply(ast.literal_eval)

In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer

X_train = train_df["text"].fillna("")
X_val = val_df["text"].fillna("")
X_test = test_df["text"].fillna("")

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

In [35]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

mlb = MultiLabelBinarizer(
    classes=list(range(28))
)

y_train = mlb.fit_transform(
    train_df["labels"]
)

y_val = mlb.transform(
    val_df["labels"]
)

y_test = mlb.transform(
    test_df["labels"]
)

print(y_train.shape)

(16531, 28)


Train baseline

In [36]:
baseline_model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    )
)

baseline_model.fit(
    X_train_tfidf,
    y_train
)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...max_iter=1000)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=

In [37]:
y_val_prob = baseline_model.predict_proba(
    X_val_tfidf
)

threshold = 0.6

y_val_pred = (
    y_val_prob >= threshold
).astype(int)

Evaluate

In [38]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

print(
    "Precision:",
    precision_score(
        y_val,
        y_val_pred,
        average="macro",
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        y_val,
        y_val_pred,
        average="macro",
        zero_division=0
    )
)

print(
    "F1:",
    f1_score(
        y_val,
        y_val_pred,
        average="macro",
        zero_division=0
    )
)

Precision: 0.3238355790172124
Recall: 0.12290679098886884
F1: 0.15753462069425364


In [39]:
y_test_prob = baseline_model.predict_proba(
    X_test_tfidf
)

y_test_pred = (
    y_test_prob >= threshold
).astype(int)

In [40]:
print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=[
            LABEL_NAMES[i]
            for i in range(28)
        ],
        zero_division=0
    )
)

                precision    recall  f1-score   support

     amusement       0.50      0.45      0.47       374
    excitement       0.37      0.41      0.39        98
           joy       0.54      0.50      0.52       204
          love       0.52      0.61      0.56       143
        desire       0.36      0.50      0.42        80
      optimism       0.57      0.76      0.65       142
        caring       0.54      0.64      0.58       150
         pride       0.71      0.64      0.67        86
    admiration       0.48      0.55      0.51       101
     gratitude       0.79      0.84      0.82       108
        relief       0.37      0.53      0.44        60
      approval       0.51      0.56      0.53       115
   realization       0.32      0.38      0.35        95
      surprise       0.45      0.49      0.47        85
     curiosity       0.45      0.53      0.49       100
     confusion       0.33      0.37      0.35        84
          fear       0.71      0.67      0.69  

In [41]:
y_pred_lr = (
    baseline_model.predict_proba(X_val_tfidf) >= threshold
).astype(int)

print("y_pred_lr:", y_pred_lr.shape)

y_pred_lr: (2066, 28)


In [42]:
from sklearn.svm import LinearSVC

svm_model = OneVsRestClassifier(
    LinearSVC(
        class_weight="balanced",
        max_iter=5000
    )
)

svm_model.fit(X_train_tfidf, y_train)

y_pred_svm = svm_model.predict(X_val_tfidf)

print("y_pred_svm:", y_pred_svm.shape)

y_pred_svm: (2066, 28)


In [43]:
emotion_names = [LABEL_NAMES[i] for i in range(28)]

print("=" * 70)
print("LOGISTIC REGRESSION")
print("=" * 70)
print(classification_report(y_val, y_pred_lr, target_names=emotion_names, zero_division=0))

print("=" * 70)
print("LINEAR SVM")
print("=" * 70)
print(classification_report(y_val, y_pred_svm, target_names=emotion_names, zero_division=0))

LOGISTIC REGRESSION
                precision    recall  f1-score   support

     amusement       0.37      0.16      0.22       327
    excitement       0.10      0.11      0.10        91
           joy       0.40      0.14      0.20       168
          love       0.20      0.17      0.18       148
        desire       0.33      0.24      0.27       102
      optimism       0.53      0.08      0.14       122
        caring       0.46      0.13      0.20       131
         pride       0.45      0.24      0.31        87
    admiration       0.33      0.05      0.08       107
     gratitude       0.67      0.19      0.29        96
        relief       0.39      0.19      0.26        58
      approval       0.23      0.26      0.25       119
   realization       0.50      0.01      0.02        96
      surprise       0.22      0.05      0.08        87
     curiosity       0.23      0.16      0.19       112
     confusion       0.22      0.26      0.24        89
          fear       0.10  

In [44]:
from sklearn.metrics import precision_recall_fscore_support

p_lr, r_lr, f_lr, sup = precision_recall_fscore_support(
    y_val, y_pred_lr, average=None, zero_division=0
)
p_svm, r_svm, f_svm, _ = precision_recall_fscore_support(
    y_val, y_pred_svm, average=None, zero_division=0
)

per_emotion = pd.DataFrame({
    "Emotion": emotion_names,
    "Support": sup,
    "P_LogReg": p_lr,
    "R_LogReg": r_lr,
    "F1_LogReg": f_lr,
    "P_SVM": p_svm,
    "R_SVM": r_svm,
    "F1_SVM": f_svm,
})

per_emotion["F1_Diff"] = per_emotion["F1_SVM"] - per_emotion["F1_LogReg"]
per_emotion["Winner"] = np.where(
    per_emotion["F1_SVM"] > per_emotion["F1_LogReg"],
    "SVM",
    "LogReg"
)

per_emotion = per_emotion.sort_values("Support", ascending=False).reset_index(drop=True)
per_emotion.round(3)

,Emotion,Support,P_LogReg,R_LogReg,F1_LogReg,P_SVM,R_SVM,F1_SVM,F1_Diff,Winner
0,sadness,354,0.438,0.161,0.236,0.267,0.404,0.322,0.086,SVM
1,annoyance,331,0.265,0.106,0.151,0.231,0.094,0.133,-0.018,LogReg
2,amusement,327,0.370,0.156,0.219,0.280,0.226,0.250,0.031,SVM
3,disappointment,211,0.259,0.033,0.059,0.145,0.047,0.071,0.013,SVM
4,anger,185,0.176,0.070,0.100,0.145,0.173,0.158,0.058,SVM
5,joy,168,0.404,0.137,0.204,0.292,0.155,0.202,-0.002,LogReg
6,disgust,164,0.189,0.146,0.165,0.196,0.165,0.179,0.014,SVM
7,love,148,0.203,0.169,0.185,0.103,0.250,0.146,-0.039,LogReg
8,embarrassment,137,0.452,0.307,0.365,0.456,0.226,0.302,-0.063,LogReg
9,caring,131,0.459,0.130,0.202,0.356,0.122,0.182,-0.021,LogReg


In [45]:
print("Number of emotions won by each model:")
print(per_emotion["Winner"].value_counts())

Number of emotions won by each model:
Winner
LogReg    15
SVM       13
Name: count, dtype: int64


In [46]:
rows = []

for name, y_pred in [
    ("TF-IDF + Logistic Regression", y_pred_lr),
    ("TF-IDF + Linear SVM", y_pred_svm),
]:
    pm, rm, fm, _ = precision_recall_fscore_support(
        y_val, y_pred, average="macro", zero_division=0
    )
    pw, rw, fw, _ = precision_recall_fscore_support(
        y_val, y_pred, average="weighted", zero_division=0
    )
    rows.append({
        "Model": name,
        "P-macro": pm,
        "R-macro": rm,
        "F1-macro": fm,
        "P-weighted": pw,
        "R-weighted": rw,
        "F1-weighted": fw,
    })

comparison = pd.DataFrame(rows).sort_values(
    "F1-macro", ascending=False
).reset_index(drop=True)

comparison.round(4)

,Model,P-macro,R-macro,F1-macro,P-weighted,R-weighted,F1-weighted
0,TF-IDF + Logistic Regression,0.3238,0.1229,0.1575,0.3225,0.1238,0.1629
1,TF-IDF + Linear SVM,0.2240,0.1428,0.1508,0.2310,0.1617,0.1666
